# Hospital Patient Volume Forecasting with Google TimesFM

## Industry: Healthcare Operations

Hospitals need reliable patient volume forecasts to avoid two costly extremes:

- **Understaffing** -> long waiting times, delayed care, overcrowding risk.
- **Overstaffing** -> unnecessary labor cost and inefficient resource allocation.

This notebook builds an end-to-end production-style forecasting workflow to predict:

- **Tomorrow's patient volume**
- **Next week's demand**
- **Holiday demand**

and convert forecasts into staffing and bed-allocation decisions.

## Data Source (Web)

We use official U.S. HHS HealthData state daily hospital utilization data (Socrata API),
filtered to **California** as a hospital-network proxy.

- Dataset page: https://healthdata.gov/Hospital/COVID-19-Reported-Patient-Impact-and-Hospital-Capa/g62h-syeh
- API endpoint used in this notebook (daily rows):
  `https://healthdata.gov/resource/g62h-syeh.csv?...`

Target series used: `inpatient_beds_used` (daily patient volume proxy).

In [ ]:
from __future__ import annotations

import math
import os
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'hospital_patient_volume'
RAW_DIR = DATA_DIR / 'raw'
ART_DIR = PROJECT_ROOT / 'artifacts' / 'hospital_patient_volume_timesfm'

RAW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_CSV = RAW_DIR / 'hhs_ca_daily_patient_volume.csv'

print('Project root:', PROJECT_ROOT)
print('Local CSV:', LOCAL_CSV)
print('Artifacts dir:', ART_DIR)

## 1) Download Daily Patient Count from Web

In [ ]:
HHS_QUERY_URL = (
    "https://healthdata.gov/resource/g62h-syeh.csv"
    "?$select=date,state,inpatient_beds_used,inpatient_beds,inpatient_beds_utilization"
    "&$where=state='CA'"
    "&$order=date"
    "&$limit=50000"
)


def load_or_download_daily_volume(local_csv: Path) -> pd.DataFrame:
    if local_csv.exists():
        print(f'Using cached file: {local_csv}')
        df = pd.read_csv(local_csv, parse_dates=['date'])
        return df

    print('Downloading from HealthData.gov API...')
    df = pd.read_csv(HHS_QUERY_URL, parse_dates=['date'])
    df.to_csv(local_csv, index=False)
    print(f'Saved downloaded file: {local_csv}')
    return df


raw_df = load_or_download_daily_volume(LOCAL_CSV)
print('Raw shape:', raw_df.shape)
raw_df.head()

## 2) Clean, Validate, and Build Forecast Series

In [ ]:
df = raw_df.copy()
df = df.rename(columns={'date': 'ds', 'inpatient_beds_used': 'patients'})

for c in ['patients', 'inpatient_beds', 'inpatient_beds_utilization']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = (
    df[['ds', 'state', 'patients', 'inpatient_beds', 'inpatient_beds_utilization']]
    .dropna(subset=['ds', 'patients'])
    .drop_duplicates(subset=['ds'])
    .sort_values('ds')
)

# Ensure daily regular grid.
full_idx = pd.date_range(df['ds'].min(), df['ds'].max(), freq='D')
df = df.set_index('ds').reindex(full_idx).rename_axis('ds').reset_index()

# Fill small gaps if any.
for c in ['patients', 'inpatient_beds', 'inpatient_beds_utilization']:
    df[c] = df[c].interpolate(limit_direction='both')

df['state'] = df['state'].fillna('CA')

print('Prepared rows:', len(df))
print('Date range:', df['ds'].min().date(), '->', df['ds'].max().date())
print('Any nulls:', bool(df.isna().any().any()))

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['ds'], df['patients'], lw=1)
ax.set_title('Daily Hospital Patient Volume (CA proxy)')
ax.set_xlabel('Date')
ax.set_ylabel('Patients (inpatient_beds_used)')
plt.tight_layout()
plt.show()

## 3) Configure Forecasting Task

We train TimesFM to forecast up to 90 days ahead so we can extract:

- tomorrow forecast (`H=1`)
- next-week forecast (`H=7`)
- holiday-demand forecasts (from horizon window)

Backtesting is rolling-origin to avoid leakage.

In [ ]:
@dataclass
class Config:
    context_len: int = 365
    max_horizon: int = 90
    eval_horizons: tuple[int, ...] = (1, 7, 30)
    # days before latest date for rolling backtest anchors
    anchor_offsets: tuple[int, ...] = (300, 240, 180, 120, 90, 60)
    per_core_batch_size: int = 8
    xreg_mode: str = 'xreg + timesfm'
    xreg_ridge: float = 1e-3


cfg = Config()

last_pos = len(df) - 1
anchor_positions: list[int] = []
for off in cfg.anchor_offsets:
    end_pos = last_pos - off
    if end_pos - cfg.context_len + 1 >= 0 and end_pos + cfg.max_horizon < len(df):
        anchor_positions.append(end_pos)

print('Anchor count:', len(anchor_positions))
print('Anchor dates:')
for pos in anchor_positions:
    print(' -', df.loc[pos, 'ds'].date())

## 4) Load TimesFM + Build Holiday/Calendar Covariates

In [ ]:
import timesfm

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
fc = timesfm.ForecastConfig(
    max_context=cfg.context_len,
    max_horizon=cfg.max_horizon,
    normalize_inputs=True,
    per_core_batch_size=cfg.per_core_batch_size,
    use_continuous_quantile_head=True,
    force_flip_invariance=True,
    infer_is_positive=True,
    fix_quantile_crossing=True,
    return_backcast=True,
)
model.compile(fc)
print('TimesFM compiled.')

In [ ]:
cal = USFederalHolidayCalendar()


def build_covariates(start_date: pd.Timestamp, context_len: int, horizon: int) -> dict[str, list[list[float]]]:
    full_dates = pd.date_range(start_date, periods=context_len + horizon, freq='D')
    dow = full_dates.dayofweek.values.astype(np.float32)
    month = full_dates.month.values.astype(np.float32)

    holiday_idx = cal.holidays(start=full_dates.min(), end=full_dates.max())
    is_holiday = full_dates.isin(holiday_idx).astype(np.float32)

    return {
        'dow_sin': [np.sin(2 * np.pi * dow / 7).astype(np.float32).tolist()],
        'dow_cos': [np.cos(2 * np.pi * dow / 7).astype(np.float32).tolist()],
        'month_sin': [np.sin(2 * np.pi * month / 12).astype(np.float32).tolist()],
        'month_cos': [np.cos(2 * np.pi * month / 12).astype(np.float32).tolist()],
        'is_weekend': [(dow >= 5).astype(np.float32).tolist()],
        'is_holiday': [is_holiday.astype(np.float32).tolist()],
    }


def forecast_patients(context: np.ndarray, start_date: pd.Timestamp, horizon: int) -> tuple[np.ndarray, np.ndarray]:
    cov = build_covariates(start_date, len(context), horizon)
    try:
        point, quant = model.forecast_with_covariates(
            inputs=[context.astype(np.float32)],
            dynamic_numerical_covariates=cov,
            xreg_mode=cfg.xreg_mode,
            ridge=cfg.xreg_ridge,
        )
    except Exception:
        point, quant = model.forecast(horizon=horizon, inputs=[context.astype(np.float32)])

    point_np = np.asarray(point, dtype=np.float32)[0, :horizon]
    quant_np = np.asarray(quant, dtype=np.float32)[0, :horizon, :]
    point_np = np.clip(point_np, 0.0, None)
    quant_np = np.clip(quant_np, 0.0, None)
    return point_np, quant_np


def wmape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.abs(y_true - y_pred).sum() / (np.abs(y_true).sum() + 1e-8))

## 5) Backtest: TimesFM vs Baselines

Baselines:
- `naive_last`: persistence
- `seasonal7`: same weekday last week pattern

In [ ]:
metric_rows: list[dict] = []
holiday_rows: list[dict] = []

for anchor_pos in anchor_positions:
    anchor_date = df.loc[anchor_pos, 'ds']
    y_ctx = df['patients'].iloc[anchor_pos - cfg.context_len + 1 : anchor_pos + 1].to_numpy(np.float32)
    y_true = df['patients'].iloc[anchor_pos + 1 : anchor_pos + cfg.max_horizon + 1].to_numpy(np.float32)

    start_date = df.loc[anchor_pos - cfg.context_len + 1, 'ds']
    pred_tfm, quant_tfm = forecast_patients(y_ctx, start_date, cfg.max_horizon)

    pred_naive = np.repeat(y_ctx[-1], cfg.max_horizon)
    pred_seasonal7 = np.tile(y_ctx[-7:], math.ceil(cfg.max_horizon / 7))[: cfg.max_horizon]

    for hz in cfg.eval_horizons:
        y = y_true[:hz]
        for model_name, p in {
            'timesfm': pred_tfm[:hz],
            'naive_last': pred_naive[:hz],
            'seasonal7': pred_seasonal7[:hz],
        }.items():
            metric_rows.append(
                {
                    'anchor_date': anchor_date,
                    'horizon_days': hz,
                    'model': model_name,
                    'mae': mean_absolute_error(y, p),
                    'rmse': mean_squared_error(y, p) ** 0.5,
                    'wmape': wmape(y, p),
                }
            )

    # Holiday-specific evaluation inside each horizon window.
    future_dates = pd.date_range(anchor_date + pd.Timedelta(days=1), periods=cfg.max_horizon, freq='D')
    holiday_dates = set(cal.holidays(start=future_dates.min(), end=future_dates.max()))
    mask_holiday = np.array([d in holiday_dates for d in future_dates])

    if mask_holiday.any():
        y_h = y_true[mask_holiday]
        p_h = pred_tfm[mask_holiday]
        holiday_rows.append(
            {
                'anchor_date': anchor_date,
                'holiday_points': int(mask_holiday.sum()),
                'holiday_mae_timesfm': mean_absolute_error(y_h, p_h),
                'holiday_wmape_timesfm': wmape(y_h, p_h),
                'avg_actual_holiday_patients': float(np.mean(y_h)),
                'avg_pred_holiday_patients': float(np.mean(p_h)),
            }
        )

metrics = (
    pd.DataFrame(metric_rows)
    .groupby(['horizon_days', 'model'], as_index=False)[['mae', 'rmse', 'wmape']]
    .mean()
    .sort_values(['horizon_days', 'rmse'])
)

holiday_eval = pd.DataFrame(holiday_rows)
metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=False)
for ax, metric in zip(axes, ['mae', 'rmse', 'wmape']):
    p = metrics.pivot(index='horizon_days', columns='model', values=metric)
    p.plot(kind='bar', ax=ax)
    ax.set_title(metric.upper())
    ax.set_xlabel('Horizon (days)')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
holiday_eval

## 6) Produce Final Operational Forecast (Tomorrow + Next Week + Holiday Demand)

In [ ]:
# Use full available history for forward forecast.
anchor_date = df['ds'].iloc[-1]
context = df['patients'].iloc[-cfg.context_len:].to_numpy(np.float32)
start_date = df['ds'].iloc[-cfg.context_len]

pred_p50, pred_quant = forecast_patients(context, start_date, cfg.max_horizon)
future_dates = pd.date_range(anchor_date + pd.Timedelta(days=1), periods=cfg.max_horizon, freq='D')

holiday_idx = set(cal.holidays(start=future_dates.min(), end=future_dates.max()))

forecast_df = pd.DataFrame(
    {
        'date': future_dates,
        'patients_p50': pred_p50,
        'patients_q10': pred_quant[:, 1],
        'patients_q90': pred_quant[:, 9],
    }
)
forecast_df['is_holiday'] = forecast_df['date'].isin(holiday_idx)

# Core business outputs
forecast_tomorrow = forecast_df.iloc[[0]].copy()
forecast_next_week = forecast_df.iloc[:7].copy()
forecast_holidays = forecast_df[forecast_df['is_holiday']].copy()

print('Tomorrow forecast:')
print(forecast_tomorrow[['date', 'patients_p50', 'patients_q90']].to_string(index=False))
print('\nNext-week avg p50 patients:', round(float(forecast_next_week['patients_p50'].mean()), 2))
print('Next-week max q90 patients:', round(float(forecast_next_week['patients_q90'].max()), 2))
print('\nHoliday demand forecasts in horizon:')
print(
    forecast_holidays[['date', 'patients_p50', 'patients_q90']].to_string(index=False)
    if len(forecast_holidays)
    else 'No holidays in next 90 days'
)


## 7) Staffing + Bed Allocation Policy

Simple policy layer (editable by operations team):

- 1 doctor per 35 patients (use `q90` for safety)
- 1 emergency nurse per 12 patients (use `q90`)
- bed allocation = `q90 * 1.05` safety buffer

This converts forecast uncertainty into robust planning actions.

In [ ]:
staff_plan = forecast_df.copy()
staff_plan['doctors_needed'] = np.ceil(staff_plan['patients_q90'] / 35.0).astype(int)
staff_plan['emergency_nurses_needed'] = np.ceil(staff_plan['patients_q90'] / 12.0).astype(int)
staff_plan['beds_to_allocate'] = np.ceil(staff_plan['patients_q90'] * 1.05).astype(int)

# Focus on actionable near-term plan.
staff_next_14 = staff_plan.head(14).copy()
holiday_staffing = staff_plan[staff_plan['is_holiday']].copy()

staff_next_14[['date', 'patients_p50', 'patients_q90', 'doctors_needed', 'emergency_nurses_needed', 'beds_to_allocate', 'is_holiday']].head(14)

In [ ]:
metrics_path = ART_DIR / 'backtest_metrics.csv'
holiday_eval_path = ART_DIR / 'holiday_backtest_eval.csv'
forecast_path = ART_DIR / 'forecast_next_90_days.csv'
staff14_path = ART_DIR / 'staffing_plan_next_14_days.csv'
holiday_staff_path = ART_DIR / 'holiday_staffing_plan.csv'

metrics.to_csv(metrics_path, index=False)
holiday_eval.to_csv(holiday_eval_path, index=False)
forecast_df.to_csv(forecast_path, index=False)
staff_next_14.to_csv(staff14_path, index=False)
holiday_staffing.to_csv(holiday_staff_path, index=False)

print('Saved:', metrics_path)
print('Saved:', holiday_eval_path)
print('Saved:', forecast_path)
print('Saved:', staff14_path)
print('Saved:', holiday_staff_path)

## Final Deliverables

This notebook provides:

1. **Tomorrow patient forecast** for immediate staffing.
2. **Next week demand forecast** for roster and bed planning.
3. **Holiday demand forecast** for surge preparation.
4. Exported CSV plans for doctors, beds, and emergency staffing.

For production, schedule this notebook/pipeline daily and route outputs to workforce management and bed-control dashboards.